# Ham ya da Spam?

🎯 Bu görevin amacı, e-postaları **spam (1)** veya **normal e-posta (0)** olarak sınıflandırmaktır.

🧹 İlk olarak, bu metin verilerine **temizleme (cleaning)** teknikleri uygulanacaktır.

👩🏻‍🔬 Ardından, temizlenmiş metinler **sayısal bir gösterime** dönüştürülecektir.

✉️ Son olarak, her bir e-postayı spam mı yoksa normal mi olduğunu sınıflandırmak için  
***Multinomial Naive Bayes*** modeli uygulanacaktır.


## (0) NTLK kütüphanesi (Doğal Dil Araç Seti)

In [1]:
# !pip install nltk

In [2]:
# nltk'yi ilk kez içe aktarırken, birkaç yerleşik kütüphaneyi de indirmemiz gerekir.

import nltk

nltk.download('stopwords')
nltk.download('punkt')      # nltk<3.9.0 için
nltk.download('punkt_tab')  # nltk>=3.9.0 için
nltk.download('wordnet')
nltk.download('omw-1.4')

[nltk_data] Downloading package stopwords to
[nltk_data]     /home/oguzhan/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt to /home/oguzhan/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /home/oguzhan/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package wordnet to /home/oguzhan/nltk_data...
[nltk_data] Downloading package omw-1.4 to /home/oguzhan/nltk_data...


True

In [3]:
import pandas as pd

df = pd.read_csv("https://d32aokrjazspmn.cloudfront.net/materials/ham_spam_emails.csv")
df.head()

,text,spam
0,Subject: naturally irresistible your corporate...,1
1,Subject: the stock trading gunslinger fanny i...,1
2,Subject: unbelievable new homes made easy im ...,1
3,Subject: 4 color printing special request add...,1
4,"Subject: do not have money , get software cds ...",1


## (1) (Metin) veri setinin temizlenmesi

Veri kümesi, ham [0] veya spam [1] olarak sınıflandırılan e-postalardan oluşur. Tahmin modelini eğitmeden önce veri kümesini temizlemeniz gerekir.

### (1.1) Noktalama İşaretlerini Kaldır

❓ Noktalama işaretlerini kaldırmak için bir işlev oluşturun. Bunu `text` sütununa uygulayın ve çıktıyı `clean_text` adlı veri çerçevesinin yeni bir sütununa ekleyin. ❓

In [6]:
import string
def remove_punctuation(text):
    for punctuation in string.punctuation:
        text=text.replace(punctuation,'')
    return text
df['clean_text']=df['text'].apply(remove_punctuation)
df[['text','clean_text']].head()

,text,clean_text
0,Subject: naturally irresistible your corporate...,Subject naturally irresistible your corporate ...
1,Subject: the stock trading gunslinger fanny i...,Subject the stock trading gunslinger fanny is...
2,Subject: unbelievable new homes made easy im ...,Subject unbelievable new homes made easy im w...
3,Subject: 4 color printing special request add...,Subject 4 color printing special request addi...
4,"Subject: do not have money , get software cds ...",Subject do not have money get software cds fr...


### (1.2) Küçük Harf

❓ Metni küçük harfe çeviren bir işlev oluşturun. Bunu `clean_text`'e uygulayın ❓

In [7]:
def lower_case(text):
    return text.lower()

df['clean_text']=df['clean_text'].apply(lower_case)
df['clean_text'].head()

0    subject naturally irresistible your corporate ...
1    subject the stock trading gunslinger  fanny is...
2    subject unbelievable new homes made easy  im w...
3    subject 4 color printing special  request addi...
4    subject do not have money  get software cds fr...
Name: clean_text, dtype: object

### (1.3) Sayıları Kaldır

❓ Metinden sayıları kaldırmak için bir işlev oluşturun. Bunu `clean_text`'e uygulayın ❓

In [8]:
import re

def remove_numbers(text):
    return re.sub(r'\d+', '', text)
df['clean_text']=df['clean_text'].apply(remove_numbers)
df['clean_text'].head

<bound method NDFrame.head of 0       subject naturally irresistible your corporate ...
1       subject the stock trading gunslinger  fanny is...
2       subject unbelievable new homes made easy  im w...
3       subject  color printing special  request addit...
4       subject do not have money  get software cds fr...
                              ...                        
5723    subject re  research and development charges t...
5724    subject re  receipts from visit  jim   thanks ...
5725    subject re  enron case study update  wow  all ...
5726    subject re  interest  david   please  call shi...
5727    subject news  aurora    update  aurora version...
Name: clean_text, Length: 5728, dtype: object>

### (1.4) StopWords'ü kaldırın

❓ Metinden durdurma kelimelerini kaldırmak için bir işlev oluşturun. Bunu `clean_text`'e uygulayın. ❓

In [10]:
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

english_stopwords=set(stopwords.words("english"))
def remove_stopwords(text):
    word_tokens=word_tokenize(text)
    clean_words=[w for w in word_tokens if w not in english_stopwords]
    return " ".join(clean_words)

df['clean_text']=df['clean_text'].apply(remove_stopwords)
df['clean_text'].head()


0    subject naturally irresistible corporate ident...
1    subject stock trading gunslinger fanny merrill...
2    subject unbelievable new homes made easy im wa...
3    subject color printing special request additio...
4    subject money get software cds software compat...
Name: clean_text, dtype: object

### (1.5) Lemmatize

❓ Metni lemmatize etmek için bir fonksiyon oluşturun. Çıktının bir kelime listesi değil, tek bir dize olduğundan emin olun. Bunu `clean_text`'e uygulayın. ❓

In [11]:
from nltk.stem import WordNetLemmatizer

lemmatizer=WordNetLemmatizer()

def lemmatize_text(text):
    word_tokens=word_tokenize(text)
    lemmatize_words=[lemmatizer.lemmatize(word,pos='v') for word in word_tokens]
    return " ".join(lemmatize_words)

df['clean_text']=df['clean_text'].apply(lemmatize_text)
df['clean_text'].head()

0    subject naturally irresistible corporate ident...
1    subject stock trade gunslinger fanny merrill m...
2    subject unbelievable new home make easy im wan...
3    subject color print special request additional...
4    subject money get software cds software compat...
Name: clean_text, dtype: object

## (2) Bag-of-Words Modellemesi

### (2.1) Metin verilerini sayılara dönüştürme

❓ `clean_text`'i varsayılan CountVectorizer ile Bag-of-Words temsiline vektörleştirin. `X_bow` olarak kaydedin. ❓

In [12]:
from sklearn.feature_extraction.text import CountVectorizer
vectorizer=CountVectorizer()
X_bow=vectorizer.fit_transform(df['clean_text'])
X_bow

<5728x29698 sparse matrix of type '<class 'numpy.int64'>'
	with 483832 stored elements in Compressed Sparse Row format>

### (2.2) Çok terimli Naive Bayes Modellemesi

❓ MultinomialNB modelini bag-of-words verileriyle çapraz doğrulayın. Modelin doğruluğunu puanlayın. ❓

In [13]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import cross_val_score

nb_model=MultinomialNB()

cv_scores=cross_val_score(nb_model,X_bow,df['spam'],cv=5,scoring='accuracy')

print(f"Multinomial Naive Bayes CV Accuracy: {cv_scores.mean():.4f}")


Multinomial Naive Bayes CV Accuracy: 0.9890


🏁 Tebrikler!

💾 Not defterinizi git add/commit/push yapmayı unutmayın...

🚀 ... ve bir sonraki challenge'a geçin!